In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('Match_Results.csv')
df.head()
df.shape

(47399, 9)

In [3]:
df.isnull().sum()
# Create result column: Home Win, Away Win, or Draw
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

df['result'] = df.apply(get_result, axis=1)
df['result'].value_counts()

result
Home Win    23239
Away Win    13378
Draw        10782
Name: count, dtype: int64

In [4]:
features = ['home_team', 'away_team', 'tournament', 'neutral']
target = 'result'

X = df[features]
y = df[target]

print(X.shape)
print(y.shape)

(47399, 4)
(47399,)


In [5]:
# For Random Forest and XGBoost - LabelEncoder
le = LabelEncoder()
X_le = X.copy()
X_le['home_team'] = le.fit_transform(X_le['home_team'])
X_le['away_team'] = le.fit_transform(X_le['away_team'])
X_le['tournament'] = le.fit_transform(X_le['tournament'])
X_le['neutral'] = X_le['neutral'].astype(int)

# For Logistic Regression - OneHotEncoder
X_ohe = pd.get_dummies(X, columns=['home_team', 'away_team', 'tournament'])
X_ohe['neutral'] = X_ohe['neutral'].astype(int)

print("LabelEncoder shape:", X_le.shape)
print("OneHotEncoder shape:", X_ohe.shape)

LabelEncoder shape: (47399, 4)
OneHotEncoder shape: (47399, 824)


In [6]:
# For Random Forest and XGBoost
X_train, X_test, y_train, y_test = train_test_split(X_le, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# For Logistic Regression - OHE
X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(X_ohe, y, test_size=0.2, random_state=42)

print("Train size OHE:", X_train_ohe.shape)
print("Test size OHE:", X_test_ohe.shape)

Train size: (37919, 4)
Test size: (9480, 4)
Train size OHE: (37919, 824)
Test size OHE: (9480, 824)


In [7]:
# Encode target for XGBoost
le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train)

# Logistic Regression - OHE
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_ohe, y_train_ohe)
lr_preds = lr_model.predict(X_test_ohe)

# Random Forest - LabelEncoder
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# XGBoost - LabelEncoder
xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                           eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train_encoded)
xgb_preds = le_target.inverse_transform(xgb_model.predict(X_test))

In [8]:
# Logistic Regression evaluation
print("Logistic Regression:")
print(confusion_matrix(y_test_ohe, lr_preds))
print(accuracy_score(y_test_ohe, lr_preds))

# Random Forest evaluation
print("\nRandom Forest:")
print(confusion_matrix(y_test, rf_preds))
print(accuracy_score(y_test, rf_preds))

# XGBoost evaluation
print("\nXGBoost:")
print(confusion_matrix(y_test, xgb_preds))
print(accuracy_score(y_test, xgb_preds))

Logistic Regression:
[[1472  132 1126]
 [ 639  117 1407]
 [ 597  142 3848]]
0.5735232067510548

Random Forest:
[[ 329    8 2393]
 [  98   11 2054]
 [ 103   13 4471]]
0.5074894514767933

XGBoost:
[[ 888   58 1784]
 [ 333   35 1795]
 [ 383   48 4156]]
0.535759493670886
